# MERRA-2 Nighttime Recovery Statistics Processing

This notebook processes hourly MERRA-2 temperature data to compute daily nighttime recovery statistics.

**Nighttime window:** 8pm-6am UTC (hours 20-23, 0-5)  
**Temporal resolution:** Daily  
**Spatial resolution:** Grid cell level (lat, lon preserved)  
**Spatial extent:** US states only (masked using region_mask.nc)  
**Time period:** 1984-2025

## Metrics Computed Per Day

For each day, we compute nighttime hour counts for temperature thresholds:

### Threshold Metrics
1. `hours_below_18_above_0` - Strong recovery hours (0°C < T < 18°C)
2. `hours_below_21_above_0` - Reasonable recovery hours (0°C < T < 21°C)
3. `hours_below_24_above_0` - Weak recovery hours (0°C < T < 24°C)
4. `hours_above_21` - Poor recovery nights (T > 21°C)
5. `hours_above_24` - Very poor recovery nights (T > 24°C)
6. `hours_below_0` - Freezing conditions (T < 0°C)
7. `hours_below_neg5` - Very cold conditions (T < -5°C)
8. `hours_below_neg10` - Extremely cold conditions (T < -10°C)

## Output

- **Directory:** `processed_nighttime_recovery/`
- **File pattern:** `nighttime_recovery_{YYYYMM}.nc`
- **Format:** NetCDF4 with dimensions (time, lat, lon)
- **Variables:** 8 metrics (daily hour counts)
- **Fill value:** -1 for non-US areas and missing data

In [1]:
# Standard imports
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from tqdm.notebook import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configuration
INPUT_DIR = Path('../daily_data') if Path('../daily_data').exists() else Path('daily_data')
OUTPUT_DIR = Path('processed_nighttime_recovery')
OUTPUT_DIR.mkdir(exist_ok=True)

# Nighttime hours (8pm-6am UTC): hours 20-23, 0-5
NIGHTTIME_HOURS = list(range(20, 24)) + list(range(0, 6))

# Rolling window size (days)
ROLLING_WINDOW = 7

# Date range
START_YEAR = 1984
END_YEAR = 2025

print(f"Input directory: {INPUT_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

In [3]:
def compute_daily_nighttime_stats(date):
    """
    Compute nighttime recovery statistics for a single day.
    
    Parameters
    ----------
    date : datetime
        Date to process
    
    Returns
    -------
    dict or None
        Dictionary with metric names as keys and 2D arrays (lat, lon) as values,
        or None if file is missing
    """
    date_str = date.strftime('%Y%m%d')
    file_path = INPUT_DIR / f'merra2_us_{date_str}.nc'
    
    if not file_path.exists():
        return None
    
    try:
        ds = xr.open_dataset(file_path)
        # Extract nighttime hours (8pm-6am = hours 20-23, 0-5)
        temp = ds.T2M.isel(time=NIGHTTIME_HOURS).values  # shape: (10 hours, lat, lon)
        ds.close()
        
        # Compute threshold counts across nighttime hours
        # Sum over time dimension (axis=0) to get counts per grid cell
        # These are integer counts, so use int16 (range 0-10)
        stats = {
            'hours_below_18_above_0': np.sum((temp < 18) & (temp > 0), axis=0).astype(np.int16),
            'hours_below_21_above_0': np.sum((temp < 21) & (temp > 0), axis=0).astype(np.int16),
            'hours_below_24_above_0': np.sum((temp < 24) & (temp > 0), axis=0).astype(np.int16),
            'hours_above_21': np.sum(temp > 21, axis=0).astype(np.int16),
            'hours_above_24': np.sum(temp > 24, axis=0).astype(np.int16),
            'hours_below_0': np.sum(temp < 0, axis=0).astype(np.int16),
            'hours_below_neg5': np.sum(temp < -5, axis=0).astype(np.int16),
            'hours_below_neg10': np.sum(temp < -10, axis=0).astype(np.int16),
        }
        
        # Validation check: ensure no value exceeds 10 hours (max nighttime hours)
        max_nighttime_hours = len(NIGHTTIME_HOURS)  # 10 hours
        for metric, values in stats.items():
            max_val = np.max(values)
            if max_val > max_nighttime_hours:
                raise ValueError(
                    f"Invalid value detected for {metric} on {date_str}: "
                    f"max value = {max_val} hours, but nighttime window is only {max_nighttime_hours} hours"
                )
        
        return stats
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [4]:
def process_month(year, month):
    """
    Process all days in a given month and save to NetCDF.
    
    Parameters
    ----------
    year : int
        Year to process
    month : int
        Month to process (1-12)
    
    Returns
    -------
    bool
        True if successful, False otherwise
    """
    output_file = OUTPUT_DIR / f'nighttime_recovery_{year}{month:02d}.nc'
    
    # Skip if output file already exists
    if os.path.exists(output_file):
        print(f"Output file {output_file.name} already exists. Skipping.")
        return True
    
    # Generate all dates for the month
    start_date = datetime(year, month, 1)
    if month == 12:
        end_date = datetime(year + 1, 1, 1) - timedelta(days=1)
    else:
        end_date = datetime(year, month + 1, 1) - timedelta(days=1)
    
    dates = pd.date_range(start_date, end_date, freq='D')
    n_days = len(dates)
    
    # Get lat/lon coordinates from first available file
    lat_coords = None
    lon_coords = None
    for date in dates:
        date_str = date.strftime('%Y%m%d')
        file_path = INPUT_DIR / f'merra2_us_{date_str}.nc'
        if file_path.exists():
            ds = xr.open_dataset(file_path)
            lat_coords = ds.lat.values
            lon_coords = ds.lon.values
            ds.close()
            break
    
    if lat_coords is None:
        print(f"No valid input files found for {year}-{month:02d}")
        return False
    
    n_lat = len(lat_coords)
    n_lon = len(lon_coords)
    
    # Load US state mask
    mask_path = Path('masks/region_mask.nc')
    if not mask_path.exists():
        print(f"Warning: Region mask file not found at {mask_path}")
        print("Processing without spatial masking.")
        us_mask = None
    else:
        try:
            ds_mask = xr.open_dataset(mask_path)
            state_mask = ds_mask.state_mask.values
            ds_mask.close()
            
            # Create boolean mask: True for US states (state_mask > 0), False otherwise
            us_mask = state_mask > 0
            
            # Verify mask dimensions match data dimensions
            if us_mask.shape != (n_lat, n_lon):
                print(f"Warning: Mask dimensions {us_mask.shape} don't match data dimensions ({n_lat}, {n_lon})")
                print("Processing without spatial masking.")
                us_mask = None
        except Exception as e:
            print(f"Warning: Error loading region mask: {e}")
            print("Processing without spatial masking.")
            us_mask = None
    
    # Initialize arrays for daily statistics
    metric_names = [
        'hours_below_18_above_0',
        'hours_below_21_above_0',
        'hours_below_24_above_0',
        'hours_above_21',
        'hours_above_24',
        'hours_below_0',
        'hours_below_neg5',
        'hours_below_neg10',
    ]
    
    # Use -1 as fill value for missing data (hours are integers 0-10)
    daily_data = {metric: np.full((n_days, n_lat, n_lon), -1, dtype=np.int16) 
                  for metric in metric_names}
    
    # Process each day
    print(f"Processing {year}-{month:02d}: {n_days} days")
    for i, date in enumerate(tqdm(dates, desc=f"{year}-{month:02d}")):
        day_stats = compute_daily_nighttime_stats(date)
        if day_stats is not None:
            for metric in metric_names:
                daily_data[metric][i, :, :] = day_stats[metric]
                
                # Apply US mask: set non-US grid cells to -1
                if us_mask is not None:
                    daily_data[metric][i, ~us_mask] = -1
    
    # Create xarray Dataset for daily data
    ds_daily = xr.Dataset(
        data_vars={metric: (['time', 'lat', 'lon'], daily_data[metric]) 
                   for metric in metric_names},
        coords={
            'time': dates,
            'lat': lat_coords,
            'lon': lon_coords,
        }
    )
    
    # Add metadata
    ds_daily.attrs['title'] = 'MERRA-2 Nighttime Recovery Statistics'
    ds_daily.attrs['description'] = 'Daily nighttime (8pm-6am UTC) temperature threshold hour counts'
    ds_daily.attrs['source'] = 'MERRA-2 M2T1NXSLV collection'
    ds_daily.attrs['nighttime_hours'] = '20-23, 0-5 (UTC)'
    ds_daily.attrs['temporal_resolution'] = 'Daily'
    ds_daily.attrs['spatial_mask'] = 'US states only (state_mask > 0)'
    ds_daily.attrs['created'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    # Add variable metadata
    metric_descriptions = {
        'hours_below_18_above_0': ('Strong recovery hours (0°C < T < 18°C)', 
                                   'Daily count of nighttime hours with temperature between 0°C and 18°C'),
        'hours_below_21_above_0': ('Reasonable recovery hours (0°C < T < 21°C)', 
                                   'Daily count of nighttime hours with temperature between 0°C and 21°C'),
        'hours_below_24_above_0': ('Weak recovery hours (0°C < T < 24°C)', 
                                   'Daily count of nighttime hours with temperature between 0°C and 24°C'),
        'hours_above_21': ('Poor recovery nights (T > 21°C)', 
                          'Daily count of nighttime hours with temperature above 21°C'),
        'hours_above_24': ('Very poor recovery nights (T > 24°C)', 
                          'Daily count of nighttime hours with temperature above 24°C'),
        'hours_below_0': ('Freezing conditions (T < 0°C)', 
                         'Daily count of nighttime hours with temperature below 0°C'),
        'hours_below_neg5': ('Very cold conditions (T < -5°C)', 
                            'Daily count of nighttime hours with temperature below -5°C'),
        'hours_below_neg10': ('Extremely cold conditions (T < -10°C)', 
                             'Daily count of nighttime hours with temperature below -10°C'),
    }
    
    # Add attributes (but not _FillValue, which goes in encoding)
    for metric, (long_name, description) in metric_descriptions.items():
        ds_daily[metric].attrs.update({
            'long_name': long_name,
            'units': 'hours',
            'description': description
        })
    
    # Save to NetCDF with compression
    # Use int16 encoding for integer hour counts (saves space vs float32)
    # _FillValue goes in encoding, not attrs
    encoding = {var: {'zlib': True, 'complevel': 4, 'dtype': 'int16', '_FillValue': -1} 
                for var in ds_daily.data_vars}
    ds_daily.to_netcdf(output_file, encoding=encoding)
    ds_daily.close()
    
    print(f"Saved: {output_file.name}")
    return True

## Test Single Month

Test processing for a single month to verify the workflow.

In [5]:
# Test with a recent month
process_month(2023, 6)

Processing 2023-06: 30 days


2023-06:   0%|          | 0/30 [00:00<?, ?it/s]

Saved: nighttime_recovery_202306.nc


True

## Process All Months (1984-2025)

Process all months in the dataset. This will skip any months that have already been processed.

In [6]:
# Process all years and months
for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        try:
            process_month(year, month)
        except Exception as e:
            print(f"Error processing {year}-{month:02d}: {e}")
            continue

Processing 1984-01: 31 days


1984-01:   0%|          | 0/31 [00:00<?, ?it/s]

Saved: nighttime_recovery_198401.nc
Processing 1984-02: 29 days


1984-02:   0%|          | 0/29 [00:00<?, ?it/s]

Saved: nighttime_recovery_198402.nc
Processing 1984-03: 31 days


...

Saved: nighttime_recovery_202510.nc
Processing 2025-11: 30 days


2025-11:   0%|          | 0/30 [00:00<?, ?it/s]

Saved: nighttime_recovery_202511.nc
Processing 2025-12: 31 days


2025-12:   0%|          | 0/31 [00:00<?, ?it/s]

Saved: nighttime_recovery_202512.nc


## Verify Output

Quick verification of the output files.

In [7]:
# List all output files
output_files = sorted(OUTPUT_DIR.glob('*.nc'))
print(f"Total output files: {len(output_files)}\n")

if output_files:
    # Inspect most recent file
    latest_file = output_files[-1]
    print(f"Inspecting: {latest_file.name}")
    ds = xr.open_dataset(latest_file)
    print(ds)
    print("\nData variables:")
    print(list(ds.data_vars))
    ds.close()

Total output files: 504

Inspecting: nighttime_recovery_202512.nc
<xarray.Dataset> Size: 10MB
Dimensions:                 (time: 31, lat: 51, lon: 95)
Coordinates:
  * time                    (time) datetime64[ns] 248B 2025-12-01 ... 2025-12-31
  * lat                     (lat) float64 408B 24.0 24.5 25.0 ... 48.0 48.5 49.0
  * lon                     (lon) float64 760B -125.0 -124.4 ... -66.88 -66.25
Data variables:
    hours_below_18_above_0  (time, lat, lon) timedelta64[ns] 1MB ...
    hours_below_21_above_0  (time, lat, lon) timedelta64[ns] 1MB ...
    hours_below_24_above_0  (time, lat, lon) timedelta64[ns] 1MB ...
    hours_above_21          (time, lat, lon) timedelta64[ns] 1MB ...
    hours_above_24          (time, lat, lon) timedelta64[ns] 1MB ...
    hours_below_0           (time, lat, lon) timedelta64[ns] 1MB ...
    hours_below_neg5        (time, lat, lon) timedelta64[ns] 1MB ...
    hours_below_neg10       (time, lat, lon) timedelta64[ns] 1MB ...
Attributes:
    title:     